In [1]:
import pandas as pd
import unicodedata
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def bersihkan_obfuscation(teks):
    if not isinstance(teks, str): return ""
    
    # Normalisasi NFKC standar (Membereskan font tipografi seperti 𝐒𝐆𝐈𝟖𝟖 -> SGI88)
    teks = unicodedata.normalize('NFKC', teks)
    
    hasil = []
    for char in teks:
        try:
            nama = unicodedata.name(char)
            # Menangkap kotak / emoji huruf seperti 🅿🆄🅻🅰🆄🆆🅸🅽
            if 'SQUARED LATIN' in nama or 'REGIONAL INDICATOR SYMBOL' in nama:
                huruf_asli = nama.split()[-1].lower()
                hasil.append(huruf_asli)
            # Menangkap huruf asing (Yunani) seperti Λ -> a
            elif 'GREEK CAPITAL LETTER LAMDA' in nama or 'GREEK SMALL LETTER LAMDA' in nama:
                hasil.append('a')
            else:
                hasil.append(char)
        except ValueError:
            # Tangkap karakter yang tidak dikenali sistem (@ jadi a)
            if char == '@':
                hasil.append('a')
            else:
                hasil.append(char)
            
    teks_baru = "".join(hasil)
    
    teks_baru = teks_baru.lower()
    
    # Hapus sisa simbol/tanda baca, sisakan HANYA huruf dan angka
    teks_baru = re.sub(r'[^a-z0-9\s]', ' ', teks_baru)
    
    # Hapus spasi ganda
    teks_baru = re.sub(r'\s+', ' ', teks_baru).strip()
    
    return teks_baru

In [3]:
nama_model = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(nama_model)

teks_kotor = "🅿🆄🅻🅰🆄🆆🅸🅽 auto sultan tentang ✌✌𝐒𝐆𝐈𝟖𝟖 dan MΛNDALIKΛ77"
teks_bersih = bersihkan_obfuscation(teks_kotor)

print("\nHASIL PEMOTONGAN KATA (TOKENIZATION)")
# Cara IndoBERT membaca teks sebelum dibersihkan
token_kotor = tokenizer.tokenize(teks_kotor)
print(f"Teks Kotor dibaca IndoBERT : {token_kotor}")

# Cara IndoBERT membaca teks setelah dibersihkan
token_bersih = tokenizer.tokenize(teks_bersih)
print(f"Teks Bersih dibaca IndoBERT : {token_bersih}")


HASIL PEMOTONGAN KATA (TOKENIZATION)
Teks Kotor dibaca IndoBERT : ['[UNK]', 'auto', 'sultan', 'tentang', '[UNK]', 'dan', '[UNK]']
Teks Bersih dibaca IndoBERT : ['pulau', '##win', 'auto', 'sultan', 'tentang', 'sg', '##i', '##88', 'dan', 'manda', '##lik', '##a', '##77']


In [4]:
file_input = '../Dataset/dataset judi.csv'
df = pd.read_csv(file_input)

print(f" Memulai pembersihan untuk {len(df)} baris data")

df['komentar_super_clean'] = df['komentar'].apply(bersihkan_obfuscation)

# Tampilkan sampel untuk memastikan hasil
kolom_tampil = ['komentar', 'komentar_clean', 'komentar_super_clean']
print("\nSAMPEL DATA SEBELUM & SESUDAH DIBERSIHKAN")
display(df.loc[[0, 3, 6], kolom_tampil])

# Simpan ke file baru
file_output = '../Dataset/dataset_judi_clean.csv'
df.to_csv(file_output, index=False)

print(f"\n Dataset tersimpan di '{file_output}'")

 Memulai pembersihan untuk 10230 baris data

SAMPEL DATA SEBELUM & SESUDAH DIBERSIHKAN


,komentar,komentar_clean,komentar_super_clean
0,Makin yakin abis baca review lain tentang ✌✌𝐒𝐆...,makin yakin abis baca review lain tentang 𝐒𝐆𝐈𝟖𝟖 .,makin yakin abis baca review lain tentang sgi88
3,░𝙈𝘼𝙉𝙐𝙏88░benar2 bikin aku jadi sultan,░𝙈𝘼𝙉𝙐𝙏88░benar2 bikin sultan,manut88 benar2 bikin aku jadi sultan
6,gua udah ga bisa bayangin hidup tanpa ALEXIS-1...,gua udah ga bisa bayangin hidup tanpa alexis-1...,gua udah ga bisa bayangin hidup tanpa alexis 1...



 Dataset tersimpan di '../Dataset/dataset_judi_clean.csv'
